# 19 · Does crediting strike rotation change the batting list?

Running between the wickets (1s, 2s and 3s against what an average batter gets in the same spot) is measured, checked for repeatability, and added to the batter skill score as a fourth part. The new score is judged on the same 52 internal checks as notebook 11, with the rule fixed in advance.

Result: running is a repeatable skill (0.67 between two halves of a player's matches), but crediting it separately keeps the same 10 batters and does not predict hidden matches better, so the batter score stays.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes and the rule fixed before running</summary>

P23 - Does crediting 'running' (strike rotation: runs from 1s, 2s and 3s) improve the batter rating? (local research)  
  
Question (user, 2026-11-14): the batter skill score V-3 credits boundaries, dots and dismissals against expected, but a two  
counts the same as a single, so keeping the board moving by running is under-credited. Is that a real loss?  
  
Rule fixed BEFORE running:  
  New candidate V-5 = V-3's three parts + a fourth part, 'running' = (runs from 1s/2s/3s) - (expected runs from 1s/2s/3s for an  
  average batter in the same spot). As in V-3, each part is weighted by its run value (regression on V-1) and by its  
  split-half reliability (how repeatable it is between two halves of a player's matches).  
  Everything else is the final batting setting (pressure-weighted, alpha 0.5, k x 2, min 60 pressure balls).  
  Judged on the same 52 splits and yardstick as P9 (rank correlation with win chance added on pressure balls in hidden  
  matches; 0.5 x random + 0.5 x rolling). V-5 replaces V-3 only if it beats it by more than one paired standard error.  
  Also reported, not used to decide: plain runs (V-1), the 2026 season, the 3,993 outside matches, Kohli's rank.  
Nothing existing is overwritten: results go to research_running_* tables.
</details>

In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
from rrlib.common import (CFG, SEED, load, load_json, load_table, save_table, save_json, get_logger, Timer, weighted_spearman,
                    split_half_reliability, excluded_reasons, excluded_players)
from rrlib import ratings as RL

log = get_logger("p23_running_test")
v = load("values")
f = load("features", columns=["ball_uid", "y_class", "over_no", "wkts_before"])
c = load("ctx_preds", columns=["ball_uid", "p_dot", "p_1", "p_2", "p_3", "p_4", "p_6", "p_wkt"])
assert (v.ball_uid.values == f.ball_uid.values).all() and (v.ball_uid.values == c.ball_uid.values).all()
comp = load_json("p05_components")
chosen = load_json("p09_summary")["chosen"]["bat"]
faced = v.faced.values.astype(bool)
yc = f.y_class.values
pb = c[["p_dot", "p_1", "p_2", "p_3", "p_4", "p_6", "p_wkt"]].values
pb = pb / pb.sum(1, keepdims=True)                     # faced-ball renormalisation, exactly as P5
run_act = np.where(np.isin(yc, [1, 2, 3]), v.runs.values, 0.0)
run_exp = pb[:, 1] + 2 * pb[:, 2] + 3 * pb[:, 3]
v["r_run_bat"] = np.where(faced, run_act - run_exp, np.nan)

# ---------------------------------------------------------------- 1. V-5 components: run value and repeatability
cols = ["r_bdry_bat", "r_dot_bat", "r_wkt_bat", "r_run_bat"]
d = v.dropna(subset=cols + ["v1_bat"])
beta = dict(zip(cols, LinearRegression(fit_intercept=False).fit(d[cols], d["v1_bat"]).coef_))
rel = {}
for cc in cols:
    r, n = split_half_reliability(d, "striker_id", "match_id", cc, min_n=60)
    rel[cc] = max(0.0, r)
v["v5_bat"] = np.where(faced, sum(rel[cc] * beta[cc] * v[cc] for cc in cols), np.nan)
# is running repeatable where the user expects it to matter most: middle overs (7-15) with 3+ wickets down?
mid = (f.over_no.between(7, 15) & (f.wkts_before >= 3)).values
dm = v[mid].dropna(subset=["r_run_bat"])
rel_mid, n_mid = split_half_reliability(dm, "striker_id", "match_id", "r_run_bat", min_n=30)
dp = v[v.is_pressure.astype(bool)].dropna(subset=["r_run_bat"])
rel_p, n_p = split_half_reliability(dp, "striker_id", "match_id", "r_run_bat", min_n=30)
info = {"beta": beta, "reliability": rel, "running_reliability_middle_overs_3plus_down": rel_mid, "players_mid": n_mid,
        "running_reliability_pressure_balls": rel_p, "players_pressure": n_p, "v3_for_reference": comp["bat"]}
log.info(f"V-5 run values {({k: round(x, 2) for k, x in beta.items()})}; reliabilities {({k: round(x, 2) for k, x in rel.items()})}")
log.info(f"running reliability: middle overs 3+ down {rel_mid:.2f} ({n_mid} players); pressure balls {rel_p:.2f} ({n_p} players)")
save_json(info, "research_running_components")

# ---------------------------------------------------------------- 2. the 52 splits (same as P9) + 2026
HOLD = CFG["cv"]["holdout_season"]
V = v[v.year < HOLD]
splits = []
for tr_years, te_year in CFG["cv"]["rolling"]:
    splits.append(("rolling", f"{tr_years[0]}-{tr_years[-1]}->{te_year}", V.year.isin(tr_years).values, (V.year == te_year).values))
mids = np.sort(V.match_id.unique())
rng = np.random.RandomState(SEED)
for i in range(CFG["cv"]["random_splits"]):
    a = set(rng.choice(mids, len(mids) // 2, replace=False))
    m = V.match_id.isin(a).values
    splits.append(("random", f"r{i}", m, ~m))
METRICS = ["v3", "v5", "v1"]


def yardstick(test):
    t = test[test.is_pressure & test.v2_bat.notna()]
    g = t.groupby("striker_id").agg(y2=("v2_bat", "mean"), y1=("v1_bat", "mean"), n=("v2_bat", "size"))
    return g[g.n >= 20]


def one(kind, name, train, test):
    y = yardstick(test)
    base_ids = set(train.loc[train.is_pressure & train.v1_bat.notna(), "striker_id"].unique())
    pool = y.index[y.index.isin(base_ids)]
    if len(pool) < 15:
        return []
    yy = y.loc[pool]
    out = []
    for met in METRICS:
        sc = RL.build_scores(train, "bat", dict(chosen, value=met), comp).score.reindex(pool).fillna(-1e9)
        top = sc.sort_values(ascending=False).index[:10]
        w = lambda col, idx: np.average(yy.loc[idx, col], weights=yy.loc[idx, "n"])
        out.append({"kind": kind, "split": name, "metric": met,
                    "rho_v2": weighted_spearman(sc.values, yy.y2.values, yy.n.values),
                    "rho_v1": weighted_spearman(sc.values, yy.y1.values, yy.n.values),
                    "top10_lift_v1": 100 * (w("y1", top) - w("y1", pool)), "top10_lift_v2": 100 * (w("y2", top) - w("y2", pool))})
    return out


with Timer(log, "52 splits + 2026 x 3 metrics"):
    jobs = [(k, n, V[tr], V[te]) for k, n, tr, te in splits] + [("holdout", "2022-25->2026", v[v.year < HOLD], v[v.year == HOLD])]
    res = Parallel(n_jobs=min(8, CFG["n_jobs"]))(delayed(one)(*j) for j in jobs)
CV = pd.DataFrame([r for x in res for r in x])
save_table(CV, "research_running_cv")
summ = {}
for met in METRICS:
    r = CV[(CV.metric == met) & (CV.kind != "holdout")]
    summ[met] = {"objective": float(0.5 * r[r.kind == "random"].rho_v2.mean() + 0.5 * r[r.kind == "rolling"].rho_v2.mean()),
                 "rho_v1": float(r.rho_v1.mean()), "top10_lift_runs": float(r.top10_lift_v1.mean()),
                 "holdout_2026_rho_v2": float(CV[(CV.metric == met) & (CV.kind == "holdout")].rho_v2.iloc[0])}
piv = CV[CV.kind == "random"].pivot(index="split", columns="metric", values="rho_v2")
for met in ("v5", "v1"):
    dd = piv[met] - piv["v3"]
    summ[met]["vs_v3_random_diff"] = float(dd.mean())
    summ[met]["vs_v3_random_se"] = float(dd.std(ddof=1) / np.sqrt(len(dd)))
    summ[met]["vs_v3_better_splits"] = int((dd > 0).sum())
    summ[met]["vs_v3_objective_diff"] = summ[met]["objective"] - summ["v3"]["objective"]
summ["decision"] = ("switch to V-5" if summ["v5"]["vs_v3_objective_diff"] > summ["v5"]["vs_v3_random_se"] else "keep V-3")
log.info("\n" + pd.DataFrame(summ).drop("decision", axis=1).round(4).to_string())
log.info(f"decision by the pre-registered rule: {summ['decision']}")

# ---------------------------------------------------------------- 3. full-data lists, Kohli, outside matches
EX = set(excluded_reasons())
names = load("players").set_index("player_id").name
final10 = load_table("final_top30_bat").head(10).player_id.tolist()
lists = {}
for met in METRICS:
    sc = RL.build_scores(v, "bat", dict(chosen, value=met), comp)
    sc = sc[~sc.index.isin(EX)].sort_values("score", ascending=False)
    lists[met] = sc.index[:10].tolist()
    rk = sc.score.rank(ascending=False)
    summ[met]["kohli_rank"] = int(rk.get(47, -1))
    summ[met]["top10"] = [names.get(p) for p in lists[met]]
    summ[met]["overlap_with_submitted"] = len(set(lists[met]) & set(final10))
    log.info(f"{met}: Kohli rank {summ[met]['kohli_rank']}; overlap with submitted {summ[met]['overlap_with_submitted']}/10; top 10 {summ[met]['top10']}")
ve = load("values_ext")
EXP = excluded_players()
base_ids = set(v.loc[v.is_pressure & v.v1_bat.notna(), "striker_id"].unique())
GROUPS = {"all public matches": np.ones(len(ve), bool), "T20 World Cups": ve.is_wc.eq(1).values,
          "franchise leagues": ve.level.isin(["BBL", "PSL", "CPL", "SA20", "ILT20", "MLC", "BPL", "LPL", "Blast"]).values,
          "full-member T20Is (incl. World Cups)": ve.level.eq("T20I-full").values}
ext = []
for gname, gm in GROUPS.items():
    dd = ve[gm & ve.is_pressure & ve.wpa_bat.notna()]
    y = dd.groupby("striker_id").agg(y1=("v1_bat", "mean"), y2=("wpa_bat", "mean"), n=("wpa_bat", "size"))
    y = y[(y.n >= 20) & y.index.isin(names.index) & y.index.isin(base_ids) & ~y.index.isin(EXP)]
    avg1, avg2 = np.average(y.y1, weights=y.n), np.average(y.y2, weights=y.n)
    for met in METRICS + ["submitted"]:
        L = final10 if met == "submitted" else lists[met]
        inp = [i for i in L if i in y.index]
        ext.append({"group": gname, "list": met, "players_in_pool": len(inp),
                    "runs_per_100": 100 * (np.average(y.loc[inp, "y1"], weights=y.loc[inp, "n"]) - avg1) if inp else np.nan,
                    "winprob_pts_per_100": 100 * (np.average(y.loc[inp, "y2"], weights=y.loc[inp, "n"]) - avg2) if inp else np.nan})
EXT = pd.DataFrame(ext)
save_table(EXT, "research_running_external")
log.info("\n" + EXT.round(2).to_string())
save_json(summ, "research_running_summary")


22:24:35 V-5 run values {'r_bdry_bat': np.float64(3.63), 'r_dot_bat': np.float64(-0.13), 'r_wkt_bat': np.float64(-5.44), 'r_run_bat': np.float64(0.66)}; reliabilities {'r_bdry_bat': 0.56, 'r_dot_bat': 0.57, 'r_wkt_bat': 0.23, 'r_run_bat': 0.67}


22:24:35 running reliability: middle overs 3+ down 0.49 (153 players); pressure balls 0.40 (138 players)


22:24:56 [52 splits + 2026 x 3 metrics] 20.6s


22:24:56 
                           v3       v5       v1
objective              0.1427   0.1343   0.1203
rho_v1                 0.1510   0.1517   0.1451
top10_lift_runs       11.6837  10.3058   9.9081
holdout_2026_rho_v2   -0.0861  -0.0941  -0.1140
vs_v3_random_diff         NaN   0.0006  -0.0085
vs_v3_random_se           NaN   0.0008   0.0029
vs_v3_better_splits       NaN  25.0000  19.0000
vs_v3_objective_diff      NaN  -0.0084  -0.0224


22:24:56 decision by the pre-registered rule: keep V-3


22:24:56 v3: Kohli rank 40; overlap with submitted 10/10; top 10 ['Abhishek Sharma', 'Vaibhav Sooryavanshi', 'Suryakumar Yadav', 'Phil Salt', 'Tim David', 'Yashasvi Jaiswal', 'Heinrich Klaasen', 'Rovman Powell', 'Josh Inglis', 'Priyansh Arya']


22:24:57 v5: Kohli rank 20; overlap with submitted 10/10; top 10 ['Abhishek Sharma', 'Suryakumar Yadav', 'Vaibhav Sooryavanshi', 'Phil Salt', 'Heinrich Klaasen', 'Tim David', 'Yashasvi Jaiswal', 'Josh Inglis', 'Rovman Powell', 'Priyansh Arya']


22:24:57 v1: Kohli rank 11; overlap with submitted 7/10; top 10 ['Abhishek Sharma', 'Tim David', 'Suryakumar Yadav', 'Phil Salt', 'Rajat Patidar', 'Vaibhav Sooryavanshi', 'Heinrich Klaasen', 'Nicholas Pooran', 'Rovman Powell', 'Jos Buttler']


22:24:58 
                                   group       list  players_in_pool  runs_per_100  winprob_pts_per_100
0                     all public matches         v3                9          1.44                 0.06
1                     all public matches         v5                9          1.44                 0.06
2                     all public matches         v1                9          6.52                 0.10
3                     all public matches  submitted                9          1.44                 0.06
4                         T20 World Cups         v3                5        -13.71                -0.14
5                         T20 World Cups         v5                5        -13.71                -0.14
6                         T20 World Cups         v1                6         -5.92                -0.15
7                         T20 World Cups  submitted                5        -13.71                -0.14
8                      franchise leagues         v3   